# 02. 모델별 학습 및 재개

Basic/PE는 500epoch, 기존 Multi 실험은 500epoch 스케줄로 284epoch까지 실행했다. 같은 명령은 last.pt에서 재개한다. live_progress.json과 tqdm으로 진행률을 표시한다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd().resolve()
assert (ROOT / "run.py").exists(), "이 공유 폴더에서 노트북을 열어 주세요."
WORK_DIR = ROOT / "work"
RUN_NAME = "shared_seed42"
def run_cli(*args):
    command = [sys.executable, str(ROOT / "run.py"), "--work-dir", str(WORK_DIR), "--run-name", RUN_NAME, *map(str, args)]
    # Pipe and flush each line so progress is visible in Jupyter.
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
        try:
            for line in child.stdout:
                print(line, end="", flush=True)
            if child.wait() != 0:
                raise RuntimeError(f"실행 실패: {child.returncode}; 위 로그를 확인하세요.")
        except KeyboardInterrupt:
            child.terminate()
            child.wait()
            raise


In [ ]:
VARIANT = "fraudgt_format"  # pe_fraudgt_format / multi_fraudgt_format
DEVICE = "cuda"
MEMORY_FRACTION = 0.35
STOP_AFTER = 284 if VARIANT == "multi_fraudgt_format" else None
RUN_TRAIN = False
if RUN_TRAIN:
    args = ["train", "--variant", VARIANT, "--device", DEVICE, "--memory-fraction", str(MEMORY_FRACTION)]
    if STOP_AFTER is not None:
        args += ["--stop-after", str(STOP_AFTER)]
    run_cli(*args)
else:
    print("GPU 여유와 입력을 확인한 뒤 RUN_TRAIN=True로 실행하세요.")

In [ ]:
import pandas as pd
history = WORK_DIR / "runs" / RUN_NAME / "models" / VARIANT / "epoch_metrics.csv"
if history.exists():
    display(pd.read_csv(history).tail(10))
else:
    print("아직 이 실행의 학습 기록이 없습니다.")